In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワーク生成エンジン (Numba JIT)
# ==========================================
@njit
def build_ba_edges_fast(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    idx = 0; target_idx = 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    chosen = np.empty(m, dtype=np.int32)
    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True
                    break
            if not already:
                chosen[c_idx] = node
                c_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx]

def build_clustered_network(N, m, weak_ties_ratio=0.002):
    N_A = N // 2
    N_B = N - N_A

    row_A, col_A = build_ba_edges_fast(N_A, m)
    row_B, col_B = build_ba_edges_fast(N_B, m)

    row_B = row_B + N_A
    col_B = col_B + N_A

    row = np.concatenate([row_A, row_B])
    col = np.concatenate([col_A, col_B])

    # 弱い紐帯 (Weak Ties: 境界ノード)
    num_weak_ties = max(1, int(N * weak_ties_ratio))
    wt_row = np.random.randint(0, N_A, num_weak_ties)
    wt_col = np.random.randint(N_A, N, num_weak_ties)

    row = np.concatenate([row, wt_row, wt_col])
    col = np.concatenate([col, wt_col, wt_row])

    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))

    deg_A = np.bincount(row_A, minlength=N_A)
    hub_A = np.argmax(deg_A)

    # 境界ノードの抽出（介入ターゲット）
    boundary_nodes = np.unique(np.concatenate([wt_row, wt_col]))

    return A_sparse, N_A, hub_A, boundary_nodes

# ==========================================
# 2. シミュレーション関数 (単体複体モデルの介入評価)
# ==========================================
def run_intervention_simulation(N, A_sparse, N_A, hub_A, boundary_nodes, params, scenario):
    dt = params['dt']
    steps = params['steps']
    T_expose = params['T_expose']

    A_current = A_sparse.copy()
    degrees = np.array(A_current.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0

    # 完全な乱数による初期化（初期値依存の排除）
    np.random.seed()
    X = np.random.uniform(0, 0.2, N).astype(np.float32)
    D = np.random.uniform(0, 0.1, N).astype(np.float32)

    v_target = np.full(N, 0.3, dtype=np.float32)
    v_target[:N_A] = 1.0

    # 2-単体(トライアド)による自己治癒ブースト項
    # 予防的介入がない場合は1.0（強化なし）
    triad_boost = np.ones(N, dtype=np.float32)

    history_D_B = []

    b_val = 0.95
    b = np.zeros(N, dtype=np.float32)

    for step in range(steps):
        # フェイクの発生
        if step < T_expose:
            b[hub_A] = b_val
        else:
            b[hub_A] = 0.0

        # ==========================================
        # 介入シナリオの分岐
        # ==========================================
        # 1. 対症療法（Subtractive: 事後BAN）
        if scenario == 'BAN' and step == T_expose:
            # ハブのエッジを完全に切断
            A_current.data[A_current.indptr[hub_A]:A_current.indptr[hub_A+1]] = 0
            # 影響を受けた列もゼロにする処理（近似的に度数をゼロに）
            degrees[hub_A] = 1.0

        # 2. 予防的治療（Additive: 2-単体注入）
        if scenario == 'Additive' and step == params['T_prevent']:
            # 初期段階で境界ノードに「安全な対話空間（トライアド）」を人工構築
            # 2-単体（面）の形成による治癒力（k_restore）の飛躍的向上をモデル化
            triad_boost[boundary_nodes] = params['triad_multiplier']

        # ==========================================

        # 同調圧力の局所伝播
        neighbor_sum = A_current.dot(X)
        P = neighbor_sum / degrees

        # SDE更新
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X_new = np.clip(X + dX, 0, 1)

        # 壊死と線維化（トポロジカル・ペナルティ + 2-単体の治癒ブースト）
        repair_capacity = np.exp(-params['gamma'] * D)
        dD = (params['k_break'] * v_target * X_new * (1 - D) -
              (params['k_restore'] * triad_boost) * (1 - X_new) * D * repair_capacity) * dt
        D_new = np.clip(D + dD, 0, 1)

        X = X_new
        D = D_new

        # マジョリティ(Cluster B)への転移ダメージを記録
        history_D_B.append(np.mean(D[N_A:]))

    return history_D_B

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [100, 1000, 10000, 100000, 1000000, 10000000]
    m = 2

    params = {
        'dt': 0.1,
        'steps': 150,
        'T_prevent': 20,     # 初期段階での予防的介入タイミング
        'T_expose': 50,      # 事後の暴露/BAN介入タイミング
        'sigma': 0.05,
        'k_break': 0.6,
        'k_restore': 0.1,
        'gamma': 6.0,
        'triad_multiplier': 5.0  # 2-単体構築による局所的治癒力の強化倍率
    }

    output_dir = "tmfg_hypothesis3_results"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        print(f"\n[{N} Nodes] ネットワーク生成中...")
        try:
            A_sparse, N_A, hub_A, boundary_nodes = build_clustered_network(N, m)

            print(f"-> シミュレーション実行中 (Baseline, BAN, Additive)")
            hist_base = run_intervention_simulation(N, A_sparse, N_A, hub_A, boundary_nodes, params, 'Baseline')
            hist_ban  = run_intervention_simulation(N, A_sparse, N_A, hub_A, boundary_nodes, params, 'BAN')
            hist_add  = run_intervention_simulation(N, A_sparse, N_A, hub_A, boundary_nodes, params, 'Additive')

            del A_sparse
            gc.collect()

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['steps']),
                'Baseline_DB': hist_base,
                'Subtractive_BAN_DB': hist_ban,
                'Additive_Prevent_DB': hist_add
            })
            csv_path = os.path.join(output_dir, f'result_N{N}.csv')
            df.to_csv(csv_path, index=False)

            # グラフの出力
            plt.figure(figsize=(10, 6))
            plt.plot(df['TimeStep'], df['Baseline_DB'], 'k-', alpha=0.5, linewidth=2, label='Baseline (No Intervention)')
            plt.plot(df['TimeStep'], df['Subtractive_BAN_DB'], 'r--', linewidth=3, label=f'Subtractive BAN (t={params["T_expose"]})')
            plt.plot(df['TimeStep'], df['Additive_Prevent_DB'], 'b-', linewidth=3, label=f'Additive Prevention (t={params["T_prevent"]})')

            plt.axvline(x=params['T_prevent'], color='b', linestyle=':', alpha=0.5, label='Prevention Timing')
            plt.axvline(x=params['T_expose'], color='r', linestyle=':', alpha=0.5, label='BAN Timing')

            plt.title(f'Intervention Asymmetry: Systemic Metastasis in Majority (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Systemic Topological Defect (D_B)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            png_path = os.path.join(output_dir, f'plot_N{N}.png')
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            plt.close()

            summary_data.append({
                'N': N,
                'Final_DB_Baseline': hist_base[-1],
                'Final_DB_BAN': hist_ban[-1],
                'Final_DB_Additive': hist_add[-1]
            })

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # サマリーとZIP化
    if len(summary_data) > 0:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'summary_H3.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_DB_Baseline'], 'k-o', alpha=0.5, label='Baseline')
        plt.plot(df_sum['N'], df_sum['Final_DB_BAN'], 'r--s', label='Subtractive (Post-BAN)')
        plt.plot(df_sum['N'], df_sum['Final_DB_Additive'], 'b-^', label='Additive (Pre-Triad)')
        plt.xscale('log')
        plt.title('Final Systemic Defect across Scales by Intervention Type')
        plt.xlabel('Sample Size N (Log Scale)')
        plt.ylabel('Final Defect Level (D_B)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()

        zip_filename = 'TMFG_Hypothesis3_Results.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)

        print(f"\n🎉 完了: 結果を '{zip_filename}' に出力しました。")

if __name__ == "__main__":
    main()


[100 Nodes] ネットワーク生成中...
-> シミュレーション実行中 (Baseline, BAN, Additive)

[1000 Nodes] ネットワーク生成中...
-> シミュレーション実行中 (Baseline, BAN, Additive)

[10000 Nodes] ネットワーク生成中...
-> シミュレーション実行中 (Baseline, BAN, Additive)

[100000 Nodes] ネットワーク生成中...
-> シミュレーション実行中 (Baseline, BAN, Additive)

[1000000 Nodes] ネットワーク生成中...
-> シミュレーション実行中 (Baseline, BAN, Additive)

[10000000 Nodes] ネットワーク生成中...
-> シミュレーション実行中 (Baseline, BAN, Additive)

🎉 完了: 結果を 'TMFG_Hypothesis3_Results.zip' に出力しました。
